In [2]:
import json, os, sys
import pandas as pd

## Train Set

Constants

In [2]:
root_json = "/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer/datasets/deepfake/dataset_json/"
#json_list = ["training_indonesian_deepfake_dataset_v2_updated","df40_train_rev","facebook_dfdc_train_reduced_rev","test_200k_24nov_merged_train"]
json_list = ["training_indonesian_deepfake_dataset_v2_updated","df40_train_fs_official","facebook_dfdc_train_reduced_rev","test_200k_24nov_merged_train"]

# datasets/deepfake/dataset_json/training_indonesian_deepfake_dataset_v2_updated.json

Get JSON

In [3]:
dict_data_json = {}
dict_list_real = {}
dict_list_fake = {}

In [ ]:
for json_now in json_list:
    print(json_now)
    path_json = os.path.join(root_json,json_now+".json")
    with open(path_json,'r') as file:
        data_json = json.load(file)
    
    dict_data_json[json_now] = data_json

    list_keys = list(data_json.keys())
    list_real = [i for i in list_keys if '/real/' in i or '/live/' in i] #real: df40,dfdc
    dict_list_real[json_now] = list_real
    print(len(list_real))
    list_fake = [i for i in list_keys if ('/fake/' in i) or '/deepfake/dfdc' in i or 'training/deepfake/' in i] #df40 => fake
    dict_list_fake[json_now] = list_fake
    print(len(list_fake))

training_indonesian_deepfake_dataset_v2_updated
87094
241221
df40_train_fs_official
22993
206662
facebook_dfdc_train_reduced_rev
16203
85499
test_200k_24nov_merged_train
129818
347394


#### Replace DF40

In [28]:
# Replace DF40 with official split
# Check Original Split
root_json_df40 = "/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer/datasets/deepfake/DF40/official_split/dataset_json"
list_fs = ["FSGAN","FaceSwap","SimSwap","InSwap","BlendFace","UniFace","MobileSwap",
"e4s","FaceDancer","DeepFaceLab"]
dict_json = {k.lower():[] for k in list_fs}
dict_json_all = {}
for i in dict_json.keys():
    for j in os.listdir(root_json_df40):
        if i in j and (j.endswith('_ff.json') or j.endswith('_cdf.json') or j.endswith('deepfacelab.json')):
            dict_json[i].append(j)
            path_json = os.path.join(root_json_df40,j)
            with open(path_json,'r') as file:
                data_json = json.load(file)
            dict_json_all.update(data_json)

In [29]:
real_images = {"train":[],"val":[],"test":[]}
fake_images = {"train":[],"val":[],"test":[]}
for k,v in dict_json_all.items():
    for v_sub in v.values():
        for subg in ['train','val','test']:
            if len(v_sub[subg])>0:
                for k_vid,v_vid in v_sub[subg].items():
                    if 'real' in v_vid['label'].lower():
                        real_images[subg].extend(v_vid['frames'])
                    if 'fake' in v_vid['label'].lower():
                        fake_images[subg].extend(v_vid['frames'])

real_images_set = {}
for k,v in real_images.items():
    real_images_set[k] = list(set(v))

real_images = real_images_set

fake_images_set = {}
for k,v in fake_images.items():
    fake_images_set[k] = list(set(v))

fake_images = fake_images_set

In [70]:
real_images_train = [i.replace("deepfakes_detection_datasets/","/mnt/ssd/datasets/deepfake/DF40/train/real/") for i in real_images["train"]]
real_images_train[:5]

['/mnt/ssd/datasets/deepfake/DF40/train/real/FaceForensics++/original_sequences/youtube/c23/frames/961/186.png',
 '/mnt/ssd/datasets/deepfake/DF40/train/real/FaceForensics++/original_sequences/youtube/c23/frames/840/649.png',
 '/mnt/ssd/datasets/deepfake/DF40/train/real/FaceForensics++/original_sequences/youtube/c23/frames/592/147.png',
 '/mnt/ssd/datasets/deepfake/DF40/train/real/FaceForensics++/original_sequences/youtube/c23/frames/247/282.png',
 '/mnt/ssd/datasets/deepfake/DF40/train/real/FaceForensics++/original_sequences/youtube/c23/frames/719/310.png']

In [71]:
fake_images_train = [i.replace("deepfakes_detection_datasets/DF40_train/","/mnt/ssd/datasets/deepfake/DF40/train/fake/") for i in fake_images["train"]]
#fake_images_train = [i.replace("uniface/frames","uniface/ff/frames") for i in fake_images_train]

dict_fake_images_train = {
    i:{'label':1,
       'method':i.split('/')[i.split('/').index('fake')+1],
       'processed_path' : i,
       'landmarks_path' : None,
        'mask_path': None,
        'angle': 0
    }
for i in fake_images_train}

dict_real_images_train = {
    i:{'label':0,
       'method':i.split('/')[i.split('/').index('real')+1],
       'processed_path' : i,
       'landmarks_path' : None,
        'mask_path': None,
        'angle': 0
    }
for i in real_images_train}
dict_data_json['df40_train_fs']=dict_fake_images_train | dict_real_images_train

In [79]:
output_json = "/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer/datasets/deepfake/dataset_json/df40_train_fs_official.json"
with open(output_json, "w") as f:
    json.dump(dict_data_json['df40_train_fs'], f, indent=4)

#### Check validity

In [5]:
for key_now,value_now in dict_data_json.items():
    list_images = [i["processed_path"] for i in value_now.values()]
    list_exist_images = [i for i in list_images if os.path.exists(i)]
    print(f"{key_now} {len(list_images)} {len(list_exist_images)}")


training_indonesian_deepfake_dataset_v2_updated 328315 328315
df40_train_fs_official 229655 229655
facebook_dfdc_train_reduced_rev 101702 101702
test_200k_24nov_merged_train 477212 477212


In [6]:
list_notexist_images = [i for i in list_images if not os.path.exists(i)]

In [7]:
list_notexist_images[:5]
set([i.split("/")[-4] for i in list_notexist_images])

set()

In [8]:
# Check by method
df_dataset = pd.DataFrame(dict_data_json['training_indonesian_deepfake_dataset_v2_updated'].values())
df_dataset.groupby("method").count()

,label,processed_path,landmarks_path,mask_path,angle
method,,,,,
fsgan,71221,71221,71221,71221,71221
live,87094,87094,87094,87094,87094
roop,90000,90000,90000,90000,90000
simswap,80000,80000,80000,80000,80000


In [9]:
# Check by method
df_dataset = pd.DataFrame(dict_data_json['facebook_dfdc_train_reduced_rev'].values())
df_dataset.groupby("method").count()

,label,processed_path,landmarks_path,mask_path,angle
method,,,,,
deepfake,85499,85499,85499,85499,85499
real,16203,16203,16203,16203,16203


In [10]:
# Check by method
df_dataset = pd.DataFrame(dict_data_json['df40_train_fs_official'].values())
df_dataset.groupby("method").count()

,label,processed_path,landmarks_path,mask_path,angle
method,,,,,
FaceForensics++,22993,22993,0,0,22993
blendface,22513,22513,0,0,22513
e4s,22779,22779,0,0,22779
facedancer,22747,22747,0,0,22747
faceswap,22852,22852,0,0,22852
fsgan,21815,21815,0,0,21815
inswap,27988,27988,0,0,27988
mobileswap,22824,22824,0,0,22824
simswap,31350,31350,0,0,31350


In [89]:
# Check by method
df_dataset = pd.DataFrame(dict_data_json['test_200k_24nov_merged_train'].values())
df_dataset.groupby("method").count()

,label,processed_path,landmarks_path,mask_path,angle,status,pred_age,score_blur_face,score_dark,score_blur_img,...,image_path,dets,head_pose,swap_method,source_name,target_name,source_pred_age,source_pred_gender,target_pred_age,target_pred_gender
method,,,,,,,,,,,,,,,,,,,,,
blendface,65462,65462,0,0,65462,0,0,0,0,0,...,65462,65462,65462,65462,65462,65462,65462,65462,65462,65462
e4s,67257,67257,0,0,67257,0,0,0,0,0,...,67257,67257,67257,67257,67257,67257,67257,67257,67257,67257
inswapper,67488,67488,0,0,67488,0,0,0,0,0,...,67488,67488,67488,67488,67488,67488,67488,67488,67488,67488
mobilefaceswap,74900,74900,0,0,74900,0,0,0,0,0,...,74900,74900,74900,74900,74900,74900,74900,74900,74900,74900
production_live,129818,129818,129818,129818,129818,129818,129818,129818,129818,129818,...,0,0,0,0,0,0,0,0,0,0
reswapper,72287,72287,0,0,72287,0,0,0,0,0,...,72287,72287,72287,72287,72287,72287,72287,72287,72287,72287


##### DF40

In [61]:
# Check Original Split
root_json = "/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer/datasets/deepfake/DF40/official_split/dataset_json"
list_fs = ["FSGAN","FaceSwap","SimSwap","InSwap","BlendFace","UniFace","MobileSwap",
"e4s","FaceDancer","DeepFaceLab"]
dict_json = {k.lower():[] for k in list_fs}
dict_json_all = {}
for i in dict_json.keys():
    for j in os.listdir(root_json):
        if i in j and (j.endswith('_ff.json') or j.endswith('_cdf.json') or j.endswith('deepfacelab.json')):
            dict_json[i].append(j)
            path_json = os.path.join(root_json,j)
            with open(path_json,'r') as file:
                data_json = json.load(file)
            dict_json_all.update(data_json)

In [10]:
real_images = {"train":[],"val":[],"test":[]}
fake_images = {"train":[],"val":[],"test":[]}
for k,v in dict_json_all.items():
    for v_sub in v.values():
        for subg in ['train','val','test']:
            if len(v_sub[subg])>0:
                for k_vid,v_vid in v_sub[subg].items():
                    if 'real' in v_vid['label'].lower():
                        real_images[subg].extend(v_vid['frames'])
                    if 'fake' in v_vid['label'].lower():
                        fake_images[subg].extend(v_vid['frames'])

real_images_set = {}
for k,v in real_images.items():
    real_images_set[k] = list(set(v))

real_images = real_images_set

fake_images_set = {}
for k,v in fake_images.items():
    fake_images_set[k] = list(set(v))

fake_images = fake_images_set

In [11]:
set([i.split("/")[2] for i in fake_images["train"]])

{'blendface',
 'e4s',
 'facedancer',
 'faceswap',
 'fsgan',
 'inswap',
 'mobileswap',
 'simswap',
 'uniface'}

In [14]:
fake_images_train = [i.replace("deepfakes_detection_datasets/DF40_train/","/mnt/ssd/datasets/deepfake/DF40/train/fake/") for i in fake_images["train"]]

In [19]:
fake_images_train[0].split('/')[fake_images_train[0].split('/').index('fake')+1]

'e4s'

In [ ]:
fake_images_train = [i.replace("deepfakes_detection_datasets/DF40_train/","/mnt/ssd/datasets/deepfake/DF40/train/fake/") for i in fake_images["train"]]
dict_fake_images_train = {
    i:{'label':1,
       'method':i.split('/')[i.split('/').index('fake')+1],
       'processed_path' : i,
       'landmarks_path' : None,
        'mask_path': None,
        'angle': 0
    }
for i in fake_images_train}

In [22]:
# Check by method DF40
#df_dataset = pd.DataFrame(dict_data_json['df40_train_rev'].values())
df_dataset = pd.DataFrame(dict_fake_images_train.values())
#df_dataset = df_dataset[df_dataset["method"].isin(["Celeb-DF-v2","FaceForensics","blendface","e4s","faceswap"
#                                                   "fsgan","inswap","mobileswap","simswap","uniface","facedancer","deepfacelab"])]
df_dataset.groupby("method").count()

,label,processed_path,landmarks_path,mask_path,angle
method,,,,,
blendface,22513,22513,0,0,22513
e4s,22779,22779,0,0,22779
facedancer,22747,22747,0,0,22747
faceswap,22852,22852,0,0,22852
fsgan,21815,21815,0,0,21815
inswap,27988,27988,0,0,27988
mobileswap,22824,22824,0,0,22824
simswap,31350,31350,0,0,31350
uniface,11794,11794,0,0,11794


In [109]:
# Check by method
df_dataset = pd.DataFrame(dict_data_json['test_200k_24nov_merged_train'].values())
df_dataset.groupby("method").count()

,label,processed_path,landmarks_path,mask_path,angle,status,pred_age,score_blur_face,score_dark,score_blur_img,...,image_path,dets,head_pose,swap_method,source_name,target_name,source_pred_age,source_pred_gender,target_pred_age,target_pred_gender
method,,,,,,,,,,,,,,,,,,,,,
blendface,65462,65462,0,0,65462,0,0,0,0,0,...,65462,65462,65462,65462,65462,65462,65462,65462,65462,65462
e4s,67257,67257,0,0,67257,0,0,0,0,0,...,67257,67257,67257,67257,67257,67257,67257,67257,67257,67257
inswapper,67488,67488,0,0,67488,0,0,0,0,0,...,67488,67488,67488,67488,67488,67488,67488,67488,67488,67488
mobilefaceswap,74900,74900,0,0,74900,0,0,0,0,0,...,74900,74900,74900,74900,74900,74900,74900,74900,74900,74900
production_live,129818,129818,129818,129818,129818,129818,129818,129818,129818,129818,...,0,0,0,0,0,0,0,0,0,0
reswapper,72287,72287,0,0,72287,0,0,0,0,0,...,72287,72287,72287,72287,72287,72287,72287,72287,72287,72287


In [107]:
df_dataset[df_dataset["method"]=='deepfake']["processed_path"].apply(lambda x:x.split("/")[3:]).iloc[0]

['datasets',
 'deepfake',
 'facebook_dfdc',
 'processes',
 'train',
 'deepfake',
 'dfdc_train_part_34',
 'gouqgigfec',
 'frames',
 'face',
 '00000.png']

In [95]:
#set(list(df_dataset[df_dataset["method"]=='live']["processed_path"].apply(lambda x:x.split("/")[10].split("_")[0])))

In [63]:
list_keys[:100]

['/mnt/ssd/datasets/deepfake/indonesian_deepfake_dataset_v2_test/training/deepfake/fsgan/tompi_abimana-aryasatya.jpg',
 '/mnt/ssd/datasets/deepfake/indonesian_deepfake_dataset_v2_test/training/deepfake/fsgan/yola_eric-thohir.jpg',
 '/mnt/ssd/datasets/deepfake/indonesian_deepfake_dataset_v2_test/training/deepfake/fsgan/pandu-sjahrir_capri-smi.jpg',
 '/mnt/ssd/datasets/deepfake/indonesian_deepfake_dataset_v2_test/training/deepfake/fsgan/william-gozali_awk.jpg',
 '/mnt/ssd/datasets/deepfake/indonesian_deepfake_dataset_v2_test/training/deepfake/fsgan/pamela_stanley-theamodels.jpg',
 '/mnt/ssd/datasets/deepfake/indonesian_deepfake_dataset_v2_test/training/deepfake/fsgan/momochan_SBY.jpg',
 '/mnt/ssd/datasets/deepfake/indonesian_deepfake_dataset_v2_test/training/deepfake/fsgan/ale-geor_nugi-wicaksono.jpg',
 '/mnt/ssd/datasets/deepfake/indonesian_deepfake_dataset_v2_test/training/deepfake/fsgan/rinald-persona_enrico-oktaviano.jpg',
 '/mnt/ssd/datasets/deepfake/indonesian_deepfake_dataset_v2_t

In [ ]:
# Verify file integrity

['/mnt/ssd/datasets/deepfake/200k_live_face_dataset/live/89614_verify_c9893057-69ce-4385-bba1-4e06eda0ec7c_a8c356df-f00b-41b0-94a9-8c248be6e58f_original.jpg.jpg',
 '/mnt/ssd/datasets/deepfake/200k_live_face_dataset/live/187566_verify_96653b16-3707-45dc-b700-103872b1f874_07f7af0c-65e8-4113-970d-9a780bbd8625_original.jpg.jpg',
 '/mnt/ssd/datasets/deepfake/200k_live_face_dataset/live/181223_verify_2cf01681-947b-43b9-a4a9-367fc68e49ca_d560d14f-cc2c-48bf-90b1-9bfb7bedede7_original.jpg.jpg',
 '/mnt/ssd/datasets/deepfake/200k_live_face_dataset/live/51275_verify_f45c7aba-63f0-4347-9778-643ea3ce6572_cba1bec8-7bb3-4c5d-bfd7-f078abad4f54_original.jpg.jpg',
 '/mnt/ssd/datasets/deepfake/200k_live_face_dataset/live/225275_verify_46445c75-05ab-435d-aacf-b11b7139a054_e625882d-faf3-40d2-b92d-6f0965e3c421_original.jpg.jpg',
 '/mnt/ssd/datasets/deepfake/200k_live_face_dataset/live/12309_verify_83f2d28c-daee-42bb-9a08-f188ae24acfb_f3a47579-3581-41ff-9193-e660eee04da9_original.jpg.jpg',
 '/mnt/ssd/datasets

In [ ]:
# DF40
# training_indonesian_deepfake_dataset_v2_updated
# 200k_24nov
# Facebook DFDC

#### Review Real Samples

#### Review Deepfake Samples

## Eval Set

In [3]:
root_json = "/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer/datasets/deepfake/dataset_json/"
#json_list = ["training_indonesian_deepfake_dataset_v2_updated","df40_train_rev","facebook_dfdc_train_reduced_rev","test_200k_24nov_merged_train"]
json_list = ["evaluation_indonesian_deepfake_dataset_v2_updated","test_200k_24nov_merged_train"]

# datasets/deepfake/dataset_json/training_indonesian_deepfake_dataset_v2_updated.json

In [4]:
dict_data_json = {}
dict_list_real = {}
dict_list_fake = {}

In [9]:
for json_now in json_list:
    print(json_now)
    path_json = os.path.join(root_json,json_now+".json")
    with open(path_json,'r') as file:
        data_json = json.load(file)
    
    dict_data_json[json_now] = data_json

    list_keys = list(data_json.keys())
    list_real = [i for i in list_keys if '/real/' in i or '/live/' in i] #real: df40,dfdc
    dict_list_real[json_now] = list_real
    print(len(list_real))
    list_fake = [i for i in list_keys if ('/fake/' in i) or '/deepfake/dfdc' in i or 'evaluation/deepfake/' in i] #df40 => fake
    dict_list_fake[json_now] = list_fake
    print(len(list_fake))

evaluation_indonesian_deepfake_dataset_v2_updated
9999
30000
test_200k_24nov_merged_train
129818
347394


#### DF40 Eval Set

In [5]:
# Replace DF40 with official split
# Check Original Split
root_json_df40 = "/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer/datasets/deepfake/DF40/official_split/dataset_json"
list_fs = ["FSGAN","FaceSwap","SimSwap","InSwap","BlendFace","UniFace","MobileSwap",
"e4s","FaceDancer","DeepFaceLab"]
dict_json = {k.lower():[] for k in list_fs}
dict_json_all = {}
for i in dict_json.keys():
    for j in os.listdir(root_json_df40):
        if i in j and (j.endswith('_ff.json') or j.endswith('_cdf.json') or j.endswith('deepfacelab.json')):
            dict_json[i].append(j)
            path_json = os.path.join(root_json_df40,j)
            with open(path_json,'r') as file:
                data_json = json.load(file)
            dict_json_all.update(data_json)

In [6]:
real_images = {"train":[],"val":[],"test":[]}
fake_images = {"train":[],"val":[],"test":[]}
for k,v in dict_json_all.items():
    for v_sub in v.values():
        for subg in ['train','val','test']:
            if len(v_sub[subg])>0:
                for k_vid,v_vid in v_sub[subg].items():
                    if 'real' in v_vid['label'].lower():
                        real_images[subg].extend(v_vid['frames'])
                    if 'fake' in v_vid['label'].lower():
                        fake_images[subg].extend(v_vid['frames'])

real_images_set = {}
for k,v in real_images.items():
    real_images_set[k] = list(set(v))

real_images = real_images_set

fake_images_set = {}
for k,v in fake_images.items():
    fake_images_set[k] = list(set(v))

fake_images = fake_images_set

In [10]:
real_images_val = [i.replace("deepfakes_detection_datasets/","/mnt/ssd/datasets/deepfake/DF40/train/real/") for i in real_images["val"]]
fake_images_val = [i.replace("deepfakes_detection_datasets/DF40/","/mnt/ssd/datasets/deepfake/DF40/eval/fake/") for i in fake_images["val"]]
real_images_val[:5]

['/mnt/ssd/datasets/deepfake/DF40/train/real/Celeb-DF-v2/Celeb-real/frames/id36_0008/268.png',
 '/mnt/ssd/datasets/deepfake/DF40/train/real/FaceForensics++/original_sequences/youtube/c23/frames/650/580.png',
 '/mnt/ssd/datasets/deepfake/DF40/train/real/Celeb-DF-v2/Celeb-real/frames/id58_0002/285.png',
 '/mnt/ssd/datasets/deepfake/DF40/train/real/FaceForensics++/original_sequences/youtube/c23/frames/386/144.png',
 '/mnt/ssd/datasets/deepfake/DF40/train/real/Celeb-DF-v2/YouTube-real/frames/00288/045.png']

In [11]:
fake_images_val[:5]

['/mnt/ssd/datasets/deepfake/DF40/eval/fake/blendface/cdf/frames/id19_id27_0001/215.png',
 '/mnt/ssd/datasets/deepfake/DF40/eval/fake/simswap/cdf/frames/id17_id16_0007/211.png',
 '/mnt/ssd/datasets/deepfake/DF40/eval/fake/inswap/ff/frames/308_388/501.png',
 '/mnt/ssd/datasets/deepfake/DF40/eval/fake/mobileswap/ff/frames/456_435/587.png',
 '/mnt/ssd/datasets/deepfake/DF40/eval/fake/fsgan/ff/frames/029_048/361.png']

In [14]:
list_nonexist_images = [i for i in fake_images_val if not os.path.exists(i)]
print(len(list_nonexist_images))
list_nonexist_images = [i for i in real_images_val if not os.path.exists(i)]
print(len(list_nonexist_images))

0
1547


228026